# 06 — Scorecard ve Cutoff Analizi

**Proje:** Credit Risk Scoring (Home Credit Default Risk)

Bu defterde mimarinin **3. katmanı** çalışıyor: `predict_proba` → **kredi skoru (300–850)** + iş kararı (kabul / red).

### Akış
1. `valid_predictions.parquet` (05'ten) yükle.
2. **PDO scorecard** kur: `score = offset + factor · log((1−p)/p)`.
3. **Skor dağılımı** + **decile band analizi** (actual vs predicted PD).
4. **KS curve** üzerinden optimal cutoff bulma.
5. **Cutoff trade-off tablosu**: approval rate, default rate, capture rate.
6. **Lift chart** — modelin random'a göre ne kadar değer kattığı.
7. Artifact'leri kaydet: scorecard params, score-band tablosu, cutoff analizi.

### Tasarım kararları
- **PDO = 20**, **base_score = 600**, **base_pd = 5%** (yani 600 puan = %5 PD; her +20 puan riski yarıya indirir).
- Skorlar `[300, 850]` aralığına clip edilir (FICO konvansiyonu).
- Cutoff seçiminde önce **KS-maksimize eden** noktayı bulup, ardından business'a uygun yuvarlak değerlerle trade-off tablosu sunarız.

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import PROCESSED_DIR
from src.scorecard import (
    ScorecardParams,
    pd_to_score,
    compute_band_analysis,
    find_optimal_cutoff,
    cutoff_metrics_table,
    classification_metrics_at_cutoff,
    compute_lift_table,
    plot_score_distribution,
    plot_band_analysis,
    plot_ks_curve,
    plot_lift,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")

print(f"Processed dir : {PROCESSED_DIR}")

## 1) Veri yükleme ve scorecard parametreleri

PDO = 20 (her 20 puan riski yarıya indirir). Base score 600 = %5 PD ankörlemesi: kredi skorlamada yaygın bir referans.

In [ ]:
preds = pd.read_parquet(PROCESSED_DIR / "valid_predictions.parquet")
print(f"valid predictions : {preds.shape}")
print(f"default rate      : {preds['TARGET'].mean():.4f}")

params = ScorecardParams(pdo=20, base_score=600, base_pd=0.05)
print("\nScorecard parameters:")
for k, v in params.as_dict().items():
    print(f"  {k:>11} = {v:.4f}" if isinstance(v, float) else f"  {k:>11} = {v}")

preds["score"] = pd_to_score(preds["pd_proba"], params)
print(f"\nscore stats : min={preds['score'].min()}  max={preds['score'].max()}  "
      f"mean={preds['score'].mean():.1f}  median={preds['score'].median():.0f}")
preds.head()

## 2) Skor dağılımı ve decile band analizi

**Sol grafik:** Default ve non-default popülasyonların skor dağılımı; ne kadar ayrışırlarsa model o kadar iyi.

**Sağ tablo:** 10 quantile band; her bandın gerçek default rate'i (`actual_pd`) ile modelin tahmini (`predicted_pd`) yan yana. Yakınlık = iyi kalibrasyon.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_score_distribution(preds, ax=axes[0])

band_df = compute_band_analysis(preds, n_bands=10)
plot_band_analysis(band_df, ax=axes[1])
plt.tight_layout()
plt.show()

band_df.round(4)

## 3) KS curve ve optimal cutoff

**KS** = `max(TPR − FPR)`. Bu noktadaki skor, default ve non-default popülasyonları en iyi ayıran eşiktir; kredi sektörünün de-facto cutoff metodu.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_ks_curve(preds["TARGET"], preds["score"], ax=ax)
plt.tight_layout()
plt.show()

opt = find_optimal_cutoff(preds["TARGET"], preds["score"])
print("Optimal cutoff (KS-maximizing):")
for k, v in opt.items():
    print(f"  {k:>10} = {v:.4f}")

## 4) Cutoff trade-off tablosu

Aynı modelle farklı politik kararlar:
- **`approval_rate`**: skoru cutoff'un üstünde olan müşterilerin oranı (kabul).
- **`approve_default_rate`**: kabul edilenler arasında gerçek default oranı (zarar).
- **`catch_rate`**: gerçek defaulter'ların ne kadarını reddedebildik.

Optimum dengeyi bulmak için KS-optimal etrafında ve yuvarlak business değerlerinde test ediyoruz.

In [ ]:
ks_cutoff = int(round(opt["threshold"]))
candidate_cutoffs = sorted({500, 550, 575, ks_cutoff, 600, 625, 650})

trade_off = cutoff_metrics_table(preds["TARGET"], preds["score"], candidate_cutoffs)
trade_off.round(4)

In [ ]:
chosen_cutoff = ks_cutoff
metrics = classification_metrics_at_cutoff(preds["TARGET"], preds["score"], chosen_cutoff)

print(f"=== Selected cutoff: {chosen_cutoff} (KS-optimal) ===\n")
tn, fp, fn, tp = metrics["confusion_matrix"].ravel()
print("Confusion matrix (rows=actual, cols=predicted)")
print(f"                 pred non-default   pred default")
print(f"  actual non-def      {tn:>8,}        {fp:>8,}")
print(f"  actual default      {fn:>8,}        {tp:>8,}")
print(
    f"\nprecision : {metrics['precision']:.4f}\n"
    f"recall    : {metrics['recall']:.4f}\n"
    f"f1        : {metrics['f1']:.4f}"
)

## 5) Lift chart

Lift = `(band'daki default rate) / (genel default rate)`. **Band-1** (en riskli %10) lift'i ne kadar yüksekse model o kadar değerli; rastgele seçimde lift = 1.

In [ ]:
lift_df = compute_lift_table(preds, n_bands=10)

fig, ax = plt.subplots(figsize=(9, 5))
plot_lift(lift_df, ax=ax)
plt.tight_layout()
plt.show()

lift_df.round(4)

## 6) Artifact'leri kaydet

- `scorecard_params.json` — pdo, base_score, base_pd, factor, offset (skor üretiminin reproducible olması için).
- `valid_scores.parquet` — `SK_ID_CURR`, `TARGET`, `pd_proba`, `score`, `band` (raporlama / monitoring için).
- `cutoff_analysis.parquet` — trade-off tablosu (business sunumu için).
- `score_bands.parquet` — decile band özet tablosu.

In [ ]:
scores_out = preds.copy()
scores_out["band"] = pd.qcut(
    scores_out["score"], q=10, labels=False, duplicates="drop"
) + 1

params_path  = PROCESSED_DIR / "scorecard_params.json"
scores_path  = PROCESSED_DIR / "valid_scores.parquet"
bands_path   = PROCESSED_DIR / "score_bands.parquet"
cutoff_path  = PROCESSED_DIR / "cutoff_analysis.parquet"

with open(params_path, "w", encoding="utf-8") as f:
    json.dump(
        {**params.as_dict(), "chosen_cutoff": int(chosen_cutoff), "ks": float(opt['ks'])},
        f, indent=2,
    )

scores_out.to_parquet(scores_path, index=False)
band_df.to_parquet(bands_path, index=False)
trade_off.to_parquet(cutoff_path, index=False)

for p in (params_path, scores_path, bands_path, cutoff_path):
    size_kb = p.stat().st_size / 1024
    suffix = "MB" if size_kb > 1024 else "KB"
    size = size_kb / 1024 if size_kb > 1024 else size_kb
    print(f"saved : {p.name:<28} ({size:.2f} {suffix})")